In [ ]:
import os
import pandas as pd

input_folder = "feature_engineered"
output_folder = "final_features"


os.makedirs(output_folder, exist_ok=True)

file_list = [f for f in os.listdir(input_folder) if f.startswith("feature_engineered_") and f.endswith(".csv")]
symbol_to_features = {}


for file_name in file_list:
    df = pd.read_csv(os.path.join(input_folder, file_name))
    feature_cols = [col for col in df.columns if col.lower() not in {"date", "symbol", "index"}]
    symbol = file_name.replace("feature_engineered_", "").replace(".csv", "")
    symbol_to_features[symbol] = set(feature_cols)

common_features = set.intersection(*symbol_to_features.values())

In [ ]:
len(common_features)

In [ ]:
id_cols = ['date', 'symbol']
final_cols = id_cols + sorted(common_features)

for file_name in file_list:
    symbol = file_name.replace("feature_engineered_", "").replace(".csv", "")
    df = pd.read_csv(os.path.join(input_folder, file_name))

    # Filter only final columns (in correct order)
    df_final = df[[col for col in final_cols if col in df.columns]]

    output_path = os.path.join(output_folder, f"final_{symbol}.csv")
    df_final.to_csv(output_path, index=False)

    print(f"✅ Saved: {output_path} with shape {df_final.shape}")

In [ ]:
import os
import pandas as pd

final_folder = "final_features"  
final_files = [f for f in os.listdir(final_folder) if f.startswith("final_") and f.endswith(".csv")]

print(f"🔍 Checking NaN values across {len(final_files)} files...\n")

nan_summary = {}

for file in final_files:
    symbol = file.replace("final_", "").replace(".csv", "")
    df = pd.read_csv(os.path.join(final_folder, file))
    
    nan_counts = df.isna().sum()
    nan_columns = nan_counts[nan_counts > 0]

    if not nan_columns.empty:
        nan_summary[symbol] = nan_columns
        print(f"⚠️ NaN detected in {symbol}:")
        print(nan_columns)
        print("-" * 40)
    else:
        print(f"✅ No NaN in {symbol}")


print("\n=== Summary ===")
print(f"{len(nan_summary)} out of {len(final_files)} symbols contain missing values.")

In [ ]:
print(f"🔄 Filling NaN values (bfill ➜ ffill) in {len(final_files)} files...\n")

for file in final_files:
    symbol = file.replace("final_", "").replace(".csv", "")
    file_path = os.path.join(final_folder, file)

    # Load file
    df = pd.read_csv(file_path)

    # Apply backfill, then forward fill
    df.fillna(method='bfill', inplace=True)
    df.fillna(method='ffill', inplace=True)

    # (Optional) Double-check for any leftover NaNs
    nan_check = df.isna().sum().sum()
    if nan_check > 0:
        print(f"⚠️ {symbol} still has {nan_check} NaNs after filling.")
    else:
        print(f"✅ {symbol}: all NaNs filled")

    # Save file (overwrite)
    df.to_csv(file_path, index=False)

In [ ]:
import pandas as pd

file_path = "final_features/final_PVB_RRC.csv"
df = pd.read_csv(file_path)

# Fill NaNs in the target column with 1.278
df['year_operation_profit_growth'].fillna(1.278, inplace=True)

# Confirm filling worked
nan_count = df['year_operation_profit_growth'].isna().sum()
if nan_count == 0:
    print("✅ All NaNs in 'year_operation_profit_growth' filled with 1.278")
else:
    print(f"⚠️ Still {nan_count} NaNs remaining")

# Save back to file
df.to_csv(file_path, index=False)